# Dataset Preparation

## Purpose

Prepare COCO image-caption pairs for caption-model training and evaluation.

Preparation includes source inventory, recorded-license filtering, download validation, duplicate detection, and reproducible split assignment.

Raw images remain outside Git. Inventory and split manifests contain metadata, source identifiers, checksums, and processing decisions.

Final-test photographs and captions are excluded from exploratory review.

In [1]:
from pathlib import Path
import sys

In [2]:
project_root = Path.cwd().resolve()
annotation_archive = (
    project_root / "feasibility" / "annotations_trainval2017.zip"
)

# Stop before creating folders if the notebook is running elsewhere.
if not (project_root / ".git").exists():
    raise RuntimeError("Run this notebook from the project root.")

if not annotation_archive.is_file():
    raise FileNotFoundError(annotation_archive)

raw_data_dir = project_root / "data" / "raw" / "coco"
manifest_dir = project_root / "manifests"

raw_data_dir.mkdir(parents=True, exist_ok=True)
manifest_dir.mkdir(parents=True, exist_ok=True)

print("Python:", sys.executable)
print("Project root:", project_root)
print("Annotation archive:", annotation_archive)
print("Raw data directory:", raw_data_dir)
print("Manifest directory:", manifest_dir)

Python: /Users/ashlabs/Projects/Masters in AI/github/image-inspired-story-creator/.venv/bin/python
Project root: /Users/ashlabs/Projects/Masters in AI/github/image-inspired-story-creator
Annotation archive: /Users/ashlabs/Projects/Masters in AI/github/image-inspired-story-creator/feasibility/annotations_trainval2017.zip
Raw data directory: /Users/ashlabs/Projects/Masters in AI/github/image-inspired-story-creator/data/raw/coco
Manifest directory: /Users/ashlabs/Projects/Masters in AI/github/image-inspired-story-creator/manifests


## Training Candidate Inventory

Source: COCO train2017 caption annotations.

Eligibility: image metadata records license ID 4 (CC BY 2.0). The inventory retains source links and caption counts without downloading
photographs or assigning splits. Recorded-license eligibility does not complete the attribution audit.

In [3]:
import json
from collections import Counter
from zipfile import ZipFile

with ZipFile(annotation_archive) as archive:
    with archive.open("annotations/captions_train2017.json") as file:
        training_metadata = json.load(file)

license_lookup = {
    record["id"]: record
    for record in training_metadata["licenses"]
}
caption_counts = Counter(
    record["image_id"]
    for record in training_metadata["annotations"]
)

training_inventory = []

# Sorting by ID makes inventory order independent of source-file ordering.
for image in sorted(training_metadata["images"], key=lambda item: item["id"]):
    if image["license"] != 4:
        continue

    training_inventory.append({
        "image_id": image["id"],
        "source_split": "train2017",
        "file_name": image["file_name"],
        "width": image["width"],
        "height": image["height"],
        "license_id": image["license"],
        "license_url": license_lookup[image["license"]]["url"],
        "coco_url": image["coco_url"],
        "flickr_url": image.get("flickr_url"),
        "caption_count": caption_counts[image["id"]],
    })

inventory_ids = {record["image_id"] for record in training_inventory}

print(f"Candidate images: {len(training_inventory):,}")
print(f"Unique image IDs: {len(inventory_ids):,}")
print(
    "Associated captions:",
    f"{sum(record['caption_count'] for record in training_inventory):,}",
)
print(
    "Candidates without captions:",
    sum(record["caption_count"] == 0 for record in training_inventory),
)
print("Previously inspected image included:", 522418 in inventory_ids)

Candidate images: 19,470
Unique image IDs: 19,470
Associated captions: 97,410
Candidates without captions: 0
Previously inspected image included: True


## Inventory Provenance

The candidate inventory is saved before image downloads and split assignment.

SHA-256 checksums identify the source archive, training annotation JSON, and serialized inventory. These checksums detect content changes; they do not independently verify the source's authenticity.

In [4]:
import hashlib


def sha256_file(path, chunk_size=1024 * 1024):
    """Calculate a file's SHA-256 digest without loading it entirely.

    Parameters
    ----------
    path : pathlib.Path
        File to hash.
    chunk_size : int
        Positive number of bytes read per iteration.

    Returns
    -------
    str
        Hexadecimal SHA-256 digest.
    """
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive.")

    digest = hashlib.sha256()
    with path.open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


annotation_member = "annotations/captions_train2017.json"

# Hash the original JSON bytes, including their original formatting.
with ZipFile(annotation_archive) as archive:
    annotation_sha256 = hashlib.sha256(
        archive.read(annotation_member)
    ).hexdigest()

inventory_bytes = (
    json.dumps(
        training_inventory,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    ) + "\n"
).encode("utf-8")

inventory_path = manifest_dir / "train2017_candidates.json"
provenance_path = manifest_dir / "train2017_candidates.provenance.json"

provenance = {
    "schema_version": 1,
    "dataset": "COCO 2017",
    "source_archive": annotation_archive.name,
    "source_archive_sha256": sha256_file(annotation_archive),
    "annotation_member": annotation_member,
    "annotation_sha256": annotation_sha256,
    "selection": {"recorded_license_id": 4},
    "image_count": len(training_inventory),
    "caption_count": sum(
        record["caption_count"] for record in training_inventory
    ),
    "inventory_file": inventory_path.name,
    "inventory_sha256": hashlib.sha256(inventory_bytes).hexdigest(),
}

# Allow identical reruns, but stop before replacing different contents.
files_to_save = {
    inventory_path: inventory_bytes,
    provenance_path: (
        json.dumps(provenance, sort_keys=True, indent=2) + "\n"
    ).encode("utf-8"),
}

for path, content in files_to_save.items():
    if path.exists() and path.read_bytes() != content:
        raise FileExistsError(
            f"Existing contents differ: {path}. Review before replacing."
        )

for path, content in files_to_save.items():
    path.write_bytes(content)

print("Inventory:", inventory_path.relative_to(project_root))
print("Provenance:", provenance_path.relative_to(project_root))
print("Archive SHA-256:", provenance["source_archive_sha256"])
print("Inventory SHA-256:", provenance["inventory_sha256"])

Inventory: manifests/train2017_candidates.json
Provenance: manifests/train2017_candidates.provenance.json
Archive SHA-256: 113a836d90195ee1f884e704da6304dfaaecff1f023f49b6ca93c4aaae470268
Inventory SHA-256: 4b97e8545cfc4cd953508082c9a0363c9fcb285d5279a90c5d361796b0e1ba11


## Pilot Download Audit

The first 20 training candidates, ordered by image ID, are used to test download access and image integrity. This is an operational sample, not a representative quality-evaluation set.

Downloaded dimensions are checked against COCO metadata. Successful files receive SHA-256 checksums. Failures are recorded without silently removing candidates from the inventory.

In [5]:
import time
from io import BytesIO
from urllib.request import urlopen

from PIL import Image


def download_candidate(record, destination_dir):
    """Download or validate one candidate image.

    Parameters
    ----------
    record : dict
        Inventory entry containing file_name, coco_url, width, and height.
    destination_dir : pathlib.Path
        Directory containing the downloaded training images.

    Returns
    -------
    dict
        File size, SHA-256 checksum, dimensions, and cache-use flag.

    Raises
    ------
    ValueError
        Filename is unsafe or dimensions differ from metadata.
    OSError
        Downloading, decoding, or saving fails.
    """
    filename = record["file_name"]
    if Path(filename).name != filename:
        raise ValueError(f"Unexpected filename: {filename}")

    image_path = destination_dir / filename
    cached = image_path.is_file()

    if cached:
        content = image_path.read_bytes()
    else:
        with urlopen(record["coco_url"], timeout=30) as response:
            content = response.read()

    # Fully decode before accepting or saving the image.
    with Image.open(BytesIO(content)) as image:
        image.load()
        dimensions = image.size

    expected = (record["width"], record["height"])
    if dimensions != expected:
        raise ValueError(f"Dimensions {dimensions}; expected {expected}")

    if not cached:
        image_path.write_bytes(content)

    return {
        "size_bytes": len(content),
        "sha256": hashlib.sha256(content).hexdigest(),
        "width": dimensions[0],
        "height": dimensions[1],
        "used_cached_file": cached,
    }


training_image_dir = raw_data_dir / "train2017"
training_image_dir.mkdir(parents=True, exist_ok=True)

pilot_results = []
started = time.perf_counter()

for record in training_inventory[:20]:
    result = {"image_id": record["image_id"]}
    try:
        result.update(download_candidate(record, training_image_dir))
        result["status"] = "ok"
    except Exception as error:
        result["status"] = "failed"
        result["error"] = f"{type(error).__name__}: {error}"
    pilot_results.append(result)

successful = sum(result["status"] == "ok" for result in pilot_results)

print(f"Successful: {successful}/{len(pilot_results)}")
print(f"Elapsed seconds: {time.perf_counter() - started:.1f}")
print(
    "Cached files reused:",
    sum(result.get("used_cached_file", False) for result in pilot_results),
)

for result in pilot_results:
    if result["status"] == "failed":
        print("FAILURE:", result["image_id"], result["error"])

Successful: 20/20
Elapsed seconds: 10.3
Cached files reused: 0


## Pilot Audit Record

The pilot audit records source inventory identity, attempted image IDs, validation results, file sizes, and image checksums.

Cached-file usage and elapsed time are reported separately because they vary between runs. The audit does not assign dataset splits.

In [6]:
pilot_audit_path = manifest_dir / "train2017_pilot_audit.json"

# Retain stable validation evidence; omit run-dependent cache flags and timing.
pilot_audit = {
    "schema_version": 1,
    "inventory_sha256": provenance["inventory_sha256"],
    "selection": "First 20 candidates in ascending image_id order",
    "attempted": len(pilot_results),
    "successful": sum(
        result["status"] == "ok" for result in pilot_results
    ),
    "results": [
        {
            key: value
            for key, value in result.items()
            if key != "used_cached_file"
        }
        for result in pilot_results
    ],
}

audit_bytes = (
    json.dumps(pilot_audit, sort_keys=True, indent=2) + "\n"
).encode("utf-8")

if pilot_audit_path.exists():
    if pilot_audit_path.read_bytes() != audit_bytes:
        raise FileExistsError(
            "Existing pilot audit differs. Review before replacing it."
        )
else:
    pilot_audit_path.write_bytes(audit_bytes)

print("Saved:", pilot_audit_path.relative_to(project_root))
print("Successful:", pilot_audit["successful"])
print("Audit SHA-256:", sha256_file(pilot_audit_path))

Saved: manifests/train2017_pilot_audit.json
Successful: 20
Audit SHA-256: 863a8e559bc115d81d74ae48aecd3f663e0af3c7bc6f5f26a9d4325f13db91ac
